# HybridGraphFNet — PascalVOC-SP Node Classification (Fixed)**Task:** Node-level semantic segmentation, 21 classes, macro F1.  **Key fix:** Eigenbasis precomputed offline and cached to disk — `eigh` never called during training.| Change from Peptides | Detail ||---|---|| Input encoder | `Linear(14→H)` (continuous features) instead of `SimpleAtomEncoder` || Readout | Per-node MLP on `[B,N,H]` — NO `GatedPooling` || Loss | Weighted `CrossEntropyLoss` (class imbalance) || Eigenbasis | Precomputed once, cached to disk, loaded per-batch || Metric | Macro F1 |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import time, os, gc
from tqdm import tqdm
from sklearn.metrics import f1_score
from torch_geometric.datasets import LRGBDataset
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_batch, to_dense_adj
from torch_geometric.data import Data
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

def set_seed(s):
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    np.random.seed(s); torch.backends.cudnn.deterministic = True

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 1. Load Base Dataset

In [ ]:
DATA_ROOT = './data'
CACHE_DIR = './eigenbasis_cache_voc_sp'
TRUNC_K   = 64   # truncated eigenbasis — validated in VRAM scaling benchmark

train_ds = LRGBDataset(root=DATA_ROOT, name='PascalVOC-SP', split='train')
val_ds   = LRGBDataset(root=DATA_ROOT, name='PascalVOC-SP', split='val')
test_ds  = LRGBDataset(root=DATA_ROOT, name='PascalVOC-SP', split='test')

sample = train_ds[0]
NUM_CLASSES   = int(max(d.y.max().item() for d in [train_ds[0], val_ds[0], test_ds[0]])) + 1
NODE_FEAT_DIM = sample.x.shape[-1]
EDGE_DIM      = sample.edge_attr.shape[-1] if (sample.edge_attr is not None and sample.edge_attr.dim() > 1) else 1

print(f'Graphs — train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}')
print(f'Sample: {sample.num_nodes} nodes, {sample.num_edges} edges')
print(f'Node feat dim: {NODE_FEAT_DIM}, Edge dim: {EDGE_DIM}, Num classes: {NUM_CLASSES}')

# Node count stats
all_n = [d.num_nodes for d in train_ds]
print(f'Node counts — min: {min(all_n)}, max: {max(all_n)}, mean: {np.mean(all_n):.1f}, median: {np.median(all_n):.0f}')

## 2. Precompute Eigenbasis — One-Time OfflineIterates every graph once, computes normalized Laplacian eigenvectors (with sign canonicalization), truncates to top-k, saves to disk.  - **Stored per-graph:** `{split}/{idx}.pt` containing `{'U': [N, k], 'n': int}`  - `A_norm` is NOT cached (cheap to reconstruct from edge_index; caching would cost ~10 GB)  - eigh failures are counted and reported, not silently swallowed

In [ ]:
# ---- Disk size estimate BEFORE committing ----
total_graphs = len(train_ds) + len(val_ds) + len(test_ds)
all_nodes    = [d.num_nodes for d in train_ds] + [d.num_nodes for d in val_ds] + [d.num_nodes for d in test_ds]
avg_n        = np.mean(all_nodes)

trunc_bytes = sum(n * TRUNC_K * 4 for n in all_nodes)  # float32
full_bytes  = sum(n * n * 4 for n in all_nodes)

print('=== Disk Size Estimates ===')
print(f'Total graphs: {total_graphs:,}')
print(f'Avg nodes/graph: {avg_n:.1f}')
print(f'')
print(f'Truncated k={TRUNC_K}:')
print(f'  Total: {trunc_bytes/1e9:.2f} GB  ({trunc_bytes/1e6:.0f} MB)')
print(f'  Per graph avg: {avg_n*TRUNC_K*4/1e3:.1f} KB')
print(f'')
print(f'Full N×N eigenbasis:')
print(f'  Total: {full_bytes/1e9:.2f} GB  ({full_bytes/1e6:.0f} MB)')
print(f'  Per graph avg: {avg_n**2*4/1e6:.1f} MB')
print(f'')
print(f'>>> Using truncated k={TRUNC_K} (saves {(full_bytes-trunc_bytes)/1e9:.1f} GB vs full)')

In [ ]:
def precompute_eigenbasis_for_split(dataset, split_name, cache_dir, k=64):
    """
    Compute truncated Laplacian eigenbasis for every graph in a split.
    Saves per-graph .pt files. Returns stats dict.
    """
    split_dir = os.path.join(cache_dir, split_name)
    os.makedirs(split_dir, exist_ok=True)

    num_graphs = len(dataset)
    eigh_failures = 0
    total_time = 0.0
    total_bytes = 0

    pbar = tqdm(range(num_graphs), desc=f'Precompute {split_name}')
    for idx in pbar:
        data = dataset[idx]
        n = data.num_nodes

        # Build adjacency + self-loops
        edge_index = data.edge_index
        # Dense adj for this single graph
        adj = torch.zeros(n, n)
        adj[edge_index[0], edge_index[1]] = 1.0
        adj = adj + torch.eye(n)  # self-loops

        # Symmetric normalized adjacency
        deg = adj.sum(dim=1)
        deg_inv_sqrt = torch.pow(deg + 1e-8, -0.5)
        D_inv_sqrt = torch.diag(deg_inv_sqrt)
        A_norm = D_inv_sqrt @ adj @ D_inv_sqrt

        # Laplacian
        L = torch.eye(n) - A_norm

        t0 = time.perf_counter()
        try:
            eigenvalues, U = torch.linalg.eigh(L)

            # Sign canonicalization
            max_abs_idx = torch.abs(U).argmax(dim=0)
            signs = torch.sign(U[max_abs_idx, torch.arange(U.size(1))])
            signs[signs == 0] = 1.0
            U = U * signs.unsqueeze(0)

        except Exception as e:
            eigh_failures += 1
            U = torch.eye(n)
            if eigh_failures <= 5:
                print(f'  WARNING: eigh failed for graph {idx} (n={n}): {e}')

        total_time += time.perf_counter() - t0

        # Truncate to top-k
        k_actual = min(k, n)
        U_trunc = U[:, :k_actual]  # [n, k_actual]

        # Save
        save_path = os.path.join(split_dir, f'{idx}.pt')
        torch.save({'U': U_trunc.clone(), 'n': n, 'k': k_actual}, save_path)
        total_bytes += os.path.getsize(save_path)

        if idx % 500 == 0:
            pbar.set_postfix({'failures': eigh_failures, 'MB': f'{total_bytes/1e6:.0f}'})

    stats = {
        'split': split_name, 'num_graphs': num_graphs,
        'eigh_failures': eigh_failures, 'time_s': total_time,
        'disk_mb': total_bytes / 1e6,
    }
    return stats

print('Precompute function defined')

In [ ]:
# ---- Run precomputation (skip if cache already exists) ----
all_stats = []
needs_precompute = False
for split_name, ds in [('train', train_ds), ('val', val_ds), ('test', test_ds)]:
    split_dir = os.path.join(CACHE_DIR, split_name)
    expected = len(ds)
    if os.path.isdir(split_dir) and len(os.listdir(split_dir)) >= expected:
        print(f'{split_name}: cache exists ({expected} files), skipping')
    else:
        needs_precompute = True
        stats = precompute_eigenbasis_for_split(ds, split_name, CACHE_DIR, k=TRUNC_K)
        all_stats.append(stats)
        print(f'  {split_name}: {stats["num_graphs"]} graphs, '
              f'{stats["eigh_failures"]} eigh failures, '
              f'{stats["time_s"]:.1f}s, {stats["disk_mb"]:.1f} MB on disk')

if all_stats:
    total_fail = sum(s['eigh_failures'] for s in all_stats)
    total_time = sum(s['time_s'] for s in all_stats)
    total_mb   = sum(s['disk_mb'] for s in all_stats)
    total_g    = sum(s['num_graphs'] for s in all_stats)
    print(f'\n=== PRECOMPUTE SUMMARY ===')
    print(f'Total graphs: {total_g:,}')
    print(f'Total eigh failures: {total_fail} / {total_g} ({100*total_fail/total_g:.2f}%)')
    print(f'Total wall-clock: {total_time:.1f}s')
    print(f'Total disk: {total_mb:.1f} MB')
    if total_fail > total_g * 0.01:
        print(f'*** WARNING: {total_fail} failures ({100*total_fail/total_g:.1f}%) — '
              f'a meaningful fraction is using identity-matrix spectral features! ***')
    else:
        print(f'Failure rate acceptable ({total_fail}/{total_g}).')
else:
    print('All splits already cached.')

## 3. Cached Dataset WrapperWraps `LRGBDataset` to load precomputed `U` from disk and attach it to each `Data` object.  PyG's DataLoader then batches `cached_U` along with node features.

In [ ]:
class CachedEigenbasisDataset:
    """Wraps LRGBDataset; loads precomputed U from disk per-graph."""
    def __init__(self, base_dataset, cache_dir, split_name):
        self.base = base_dataset
        self.split_dir = os.path.join(cache_dir, split_name)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        data = self.base[idx].clone()
        cache = torch.load(os.path.join(self.split_dir, f'{idx}.pt'), weights_only=True)
        U = cache['U']  # [N, k_actual]
        # Pad to TRUNC_K if k_actual < TRUNC_K (very small graphs)
        if U.size(1) < TRUNC_K:
            U = F.pad(U, (0, TRUNC_K - U.size(1)))
        data.cached_U = U  # [N, TRUNC_K] — batched by PyG along node dim
        return data


def make_loaders(batch_size=4):
    train_cached = CachedEigenbasisDataset(train_ds, CACHE_DIR, 'train')
    val_cached   = CachedEigenbasisDataset(val_ds,   CACHE_DIR, 'val')
    test_cached  = CachedEigenbasisDataset(test_ds,  CACHE_DIR, 'test')

    train_loader = DataLoader(train_cached, batch_size=batch_size, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_cached,   batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_cached,  batch_size=batch_size, shuffle=False, num_workers=0)
    return train_loader, val_loader, test_loader


# Sanity check
_tl, _vl, _tl2 = make_loaders(batch_size=2)
_batch = next(iter(_tl))
print(f'Batch keys: {list(_batch.keys())}')
print(f'x: {_batch.x.shape}, cached_U: {_batch.cached_U.shape}, batch: {_batch.batch.shape}')
print(f'cached_U dtype: {_batch.cached_U.dtype}')

# Verify to_dense_batch works on cached_U
U_dense, U_mask = to_dense_batch(_batch.cached_U, _batch.batch)
print(f'U_dense (after to_dense_batch): {U_dense.shape}  — should be [B, N_max, {TRUNC_K}]')
del _tl, _vl, _tl2, _batch

## 4. Model Components (backbone identical to Peptides)

In [ ]:
class DenseGCNLayer(nn.Module):
    def __init__(self, hidden_dim, edge_dim=1):
        super().__init__()
        self.node_lin = nn.Linear(hidden_dim, hidden_dim)
        self.edge_lin = nn.Linear(edge_dim, 1)
        self.norm     = nn.LayerNorm(hidden_dim)

    def forward(self, x, A_norm, edge_attr_dense=None):
        if edge_attr_dense is not None:
            E   = torch.sigmoid(self.edge_lin(edge_attr_dense)).squeeze(-1)
            msg = torch.bmm(A_norm * E, x)
        else:
            msg = torch.bmm(A_norm, x)
        return self.norm(F.gelu(self.node_lin(msg)))


class SpectralMixMH(nn.Module):
    """Works with both full [B,N,N] and truncated [B,N,k] eigenbasis."""
    def __init__(self, hidden_dim, num_heads=4):
        super().__init__()
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.filter_gen = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj   = nn.Linear(hidden_dim, hidden_dim)
        self.norm       = nn.LayerNorm(hidden_dim)

    def forward(self, x, U, mask):
        # U: [B, N, K] where K can be N (full) or k (truncated)
        x_hat      = torch.bmm(U.transpose(1, 2), x)       # [B, K, H]
        fil        = torch.sigmoid(self.filter_gen(x_hat))  # [B, K, H]
        x_filtered = fil * x_hat
        x_out      = torch.bmm(U, x_filtered)               # [B, N, H]
        x_out      = x_out * mask.unsqueeze(-1)
        return self.norm(self.out_proj(F.gelu(x_out)))


print('Components defined (SpectralMixMH handles truncated U natively)')

## 5. HybridGraphFNet — Node-Level Head, Precomputed Eigenbasis**Key difference from Peptides version:**- `forward()` receives precomputed `U` via `data.cached_U` — no `eigh` called- `compute_A_norm()` is cheap O(N²) normalization only — no eigendecomposition- Readout is per-node MLP, not `GatedPooling`

In [ ]:
class HybridGraphFNet_NodeLevel(nn.Module):
    def __init__(self, in_dim=14, hidden_dim=128, num_layers=4, num_classes=21,
                 num_heads=4, lap_k=8, dropout=0.1, edge_dim=1):
        super().__init__()
        self.lap_k   = lap_k
        self.dropout = nn.Dropout(dropout)

        self.input_proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
        )
        self.pe_encoder = nn.Linear(lap_k, hidden_dim)

        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'local':  DenseGCNLayer(hidden_dim, edge_dim=edge_dim),
                'global': SpectralMixMH(hidden_dim, num_heads=num_heads),
                'gate':   nn.Linear(hidden_dim, hidden_dim),
                'norm':   nn.LayerNorm(hidden_dim),
            }) for _ in range(num_layers)
        ])

        self.node_classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, num_classes),
        )

    def compute_A_norm(self, adj, mask):
        """Cheap O(N^2) normalized adjacency. No eigendecomposition."""
        B, N, _ = adj.shape
        A_list = []
        for b in range(B):
            n = int(mask[b].sum().item())
            adj_b = adj[b, :n, :n]
            deg = adj_b.sum(dim=1)
            deg_inv_sqrt = torch.pow(deg + 1e-8, -0.5)
            D_inv_sqrt = torch.diag(deg_inv_sqrt)
            A_norm_b = D_inv_sqrt @ adj_b @ D_inv_sqrt
            A_list.append(F.pad(A_norm_b, (0, N-n, 0, N-n)))
        return torch.stack(A_list)

    def forward(self, data):
        # ---- Dense conversion ----
        x, mask = to_dense_batch(data.x.float(), data.batch)
        adj = to_dense_adj(data.edge_index, data.batch, max_num_nodes=x.size(1))

        # Edge attributes
        if data.edge_attr is not None:
            ea = data.edge_attr.float()
            if ea.dim() == 1: ea = ea.unsqueeze(-1)
            edge_attr_dense = to_dense_adj(
                data.edge_index, data.batch, edge_attr=ea, max_num_nodes=x.size(1))
        else:
            edge_attr_dense = None

        adj = adj + torch.eye(adj.size(1), device=x.device).unsqueeze(0)  # self-loops

        # ---- A_norm (cheap, no eigh) ----
        A_norm = self.compute_A_norm(adj, mask)

        # ---- Precomputed eigenbasis from cache ----
        U, _ = to_dense_batch(data.cached_U.float(), data.batch)  # [B, N_max, k]
        U = U * mask.unsqueeze(-1)  # zero padded positions

        # ---- Node features ----
        x = self.input_proj(x)

        # ---- LapPE injection (first lap_k columns of cached U) ----
        k = min(self.lap_k, U.size(-1))
        lap_pe = U[:, :, :k] * mask.unsqueeze(-1)
        x = x + self.pe_encoder(lap_pe)

        # ---- Message passing (IDENTICAL to Peptides) ----
        for layer in self.layers:
            x_res    = x
            x_local  = layer['local'](x, A_norm, edge_attr_dense)
            x_global = layer['global'](x, U, mask)
            gate     = torch.sigmoid(layer['gate'](x))
            x_mix    = gate * x_local + (1 - gate) * x_global
            x        = layer['norm'](x_res + self.dropout(x_mix))

        # ---- Per-node classification (NO GatedPooling) ----
        x = x * mask.unsqueeze(-1)
        logits = self.node_classifier(x)  # [B, N, num_classes]
        return logits, mask


# Sanity check
set_seed(0)
model = HybridGraphFNet_NodeLevel(
    in_dim=NODE_FEAT_DIM, hidden_dim=128, num_layers=4,
    num_classes=NUM_CLASSES, edge_dim=EDGE_DIM,
).to(device)

print(f'Parameters: {count_params(model):,}')
_tl, _, _ = make_loaders(batch_size=2)
_b = next(iter(_tl)).to(device)
with torch.no_grad():
    logits, mask = model(_b)
print(f'Forward pass: logits={logits.shape}, mask={mask.shape}, valid_nodes={mask.sum().item()}')
print('No eigh called during forward pass.')
del _tl, _b

## 6. Evaluation: Macro F1

In [ ]:
def evaluate_macro_f1(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits, mask = model(batch)
            y_dense, _ = to_dense_batch(batch.y, batch.batch)
            valid = mask.bool()
            all_preds.append(logits[valid].argmax(dim=-1).cpu())
            all_labels.append(y_dense[valid].cpu())
    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    return f1_score(all_labels, all_preds, average='macro', zero_division=0)

print('evaluate_macro_f1 defined')

## 7. Gate Health Check (uses precomputed U path)

In [ ]:
def check_gate_health(model, val_loader, device):
    model.eval()
    batch = next(iter(val_loader)).to(device)
    with torch.no_grad():
        x, mask = to_dense_batch(batch.x.float(), batch.batch)
        adj = to_dense_adj(batch.edge_index, batch.batch, max_num_nodes=x.size(1))
        adj = adj + torch.eye(adj.size(1), device=x.device).unsqueeze(0)

        # Use precomputed U (same path as training)
        U, _ = to_dense_batch(batch.cached_U.float(), batch.batch)
        U = U * mask.unsqueeze(-1)

        x_enc = model.input_proj(x)
        k = min(model.lap_k, U.size(-1))
        lap_pe = U[:, :, :k] * mask.unsqueeze(-1)
        x_enc = x_enc + model.pe_encoder(lap_pe)

        print('\n--- Gate Health Check ---')
        collapsed = False
        for i, layer in enumerate(model.layers):
            gv = torch.sigmoid(layer['gate'](x_enc))
            mg, sg = gv.mean().item(), gv.std().item()
            if mg > 0.85:   status = 'COLLAPSED->GCN';      collapsed = True
            elif mg < 0.15: status = 'COLLAPSED->SPECTRAL';  collapsed = True
            elif sg < 0.05: status = 'UNIFORM';              collapsed = True
            else:           status = 'HEALTHY'
            print(f'  Layer {i} | mean={mg:.4f} | std={sg:.4f} | {status}')
        if collapsed: print('  ACTION: lr will be reset.')
        else:         print('  All gates healthy.')
        print('-------------------------\n')
    model.train()
    return collapsed

print('check_gate_health defined (uses precomputed-U path)')

## 8. Timing Comparison: Live eigh vs PrecomputedRuns N batches with the OLD path (eigh every forward pass) vs NEW path (precomputed U).  Provides a concrete, citable number for the paper.

In [ ]:
def compute_laplacian_basis_LIVE(adj, mask):
    """OLD path — eigh called per-graph per-batch. For timing comparison only."""
    B, N, _ = adj.shape
    A_list, U_list = [], []
    for b in range(B):
        n = int(mask[b].sum().item())
        adj_b = adj[b, :n, :n]
        deg = adj_b.sum(dim=1)
        deg_inv_sqrt = torch.pow(deg + 1e-8, -0.5)
        D_inv_sqrt = torch.diag(deg_inv_sqrt)
        A_norm_b = D_inv_sqrt @ adj_b @ D_inv_sqrt
        L_b = torch.eye(n, device=adj.device) - A_norm_b
        try:
            _, U_b = torch.linalg.eigh(L_b)
            max_abs_idx = torch.abs(U_b).argmax(dim=0)
            signs = torch.sign(U_b[max_abs_idx, torch.arange(n, device=adj.device)])
            signs[signs == 0] = 1.0
            U_b = U_b * signs.unsqueeze(0)
        except Exception:
            U_b = torch.eye(n, device=adj.device)
        A_list.append(F.pad(A_norm_b, (0, N-n, 0, N-n)))
        U_list.append(F.pad(U_b, (0, N-n, 0, N-n)))
    return torch.stack(A_list), torch.stack(U_list)


NUM_TIMING_BATCHES = 20
BATCH_SIZE_TIMING  = 4

_tl, _, _ = make_loaders(batch_size=BATCH_SIZE_TIMING)
_batches = [next(iter(_tl)).to(device) for _ in range(min(NUM_TIMING_BATCHES, len(_tl)))]
# reload since we consumed the iterator
_tl2, _, _ = make_loaders(batch_size=BATCH_SIZE_TIMING)
_it = iter(_tl2)
_batches = []
for i in range(NUM_TIMING_BATCHES):
    try: _batches.append(next(_it).to(device))
    except StopIteration: break

model_timing = HybridGraphFNet_NodeLevel(
    in_dim=NODE_FEAT_DIM, hidden_dim=128, num_layers=4,
    num_classes=NUM_CLASSES, edge_dim=EDGE_DIM,
).to(device).eval()

# ---- OLD path: live eigh ----
t_old_total = 0.0
with torch.no_grad():
    for batch in _batches:
        x, mask = to_dense_batch(batch.x.float(), batch.batch)
        adj = to_dense_adj(batch.edge_index, batch.batch, max_num_nodes=x.size(1))
        adj = adj + torch.eye(adj.size(1), device=x.device).unsqueeze(0)
        t0 = time.perf_counter()
        A_norm, U = compute_laplacian_basis_LIVE(adj, mask)
        if device.type == 'cuda': torch.cuda.synchronize()
        t_old_total += time.perf_counter() - t0

# ---- NEW path: precomputed U ----
t_new_total = 0.0
with torch.no_grad():
    for batch in _batches:
        x, mask = to_dense_batch(batch.x.float(), batch.batch)
        adj = to_dense_adj(batch.edge_index, batch.batch, max_num_nodes=x.size(1))
        adj = adj + torch.eye(adj.size(1), device=x.device).unsqueeze(0)
        t0 = time.perf_counter()
        A_norm = model_timing.compute_A_norm(adj, mask)
        U_pre, _ = to_dense_batch(batch.cached_U.float(), batch.batch)
        if device.type == 'cuda': torch.cuda.synchronize()
        t_new_total += time.perf_counter() - t0

nb = len(_batches)
t_old_per = t_old_total / nb
t_new_per = t_new_total / nb
batches_per_epoch = len(train_ds) // BATCH_SIZE_TIMING

print(f'=== TIMING COMPARISON ({nb} batches, BS={BATCH_SIZE_TIMING}) ===')
print(f'OLD (live eigh):     {t_old_per*1000:.1f} ms/batch')
print(f'NEW (precomputed U): {t_new_per*1000:.1f} ms/batch')
print(f'Speedup per batch:   {t_old_per/t_new_per:.1f}x')
print(f'')
print(f'Projected per-epoch savings:')
print(f'  Batches/epoch: ~{batches_per_epoch}')
print(f'  OLD: ~{t_old_per * batches_per_epoch:.1f}s/epoch on eigh alone')
print(f'  NEW: ~{t_new_per * batches_per_epoch:.1f}s/epoch on A_norm + U loading')
print(f'  Saved: ~{(t_old_per - t_new_per) * batches_per_epoch:.1f}s/epoch')

del model_timing, _batches, _tl, _tl2

## 9. Memory & Timing Smoke Test**Run this BEFORE the full 3-seed sweep.** Reports peak VRAM and per-batch time at real batch_size.  If too close to VRAM limit or too slow, prints guidance.

In [ ]:
BATCH_SIZE = 4
ACCUM_STEPS = 4

_tl, _, _ = make_loaders(batch_size=BATCH_SIZE)
_model = HybridGraphFNet_NodeLevel(
    in_dim=NODE_FEAT_DIM, hidden_dim=128, num_layers=4,
    num_classes=NUM_CLASSES, edge_dim=EDGE_DIM,
).to(device)
_model.train()

_optimizer = optim.AdamW(_model.parameters(), lr=1e-3)
_criterion = nn.CrossEntropyLoss(ignore_index=-1)

if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()

_batch = next(iter(_tl)).to(device)
t0 = time.perf_counter()

# Forward
logits, mask = _model(_batch)
y_dense, _ = to_dense_batch(_batch.y, _batch.batch, fill_value=-1)
loss = _criterion(logits.reshape(-1, logits.size(-1)), y_dense.reshape(-1).long())

# Backward
loss.backward()
torch.nn.utils.clip_grad_norm_(_model.parameters(), max_norm=1.0)
_optimizer.step()
_optimizer.zero_grad()

if device.type == 'cuda': torch.cuda.synchronize()
t_batch = time.perf_counter() - t0

if device.type == 'cuda':
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
else:
    peak_gb = 0.0; total_gb = 16.0

batches_per_epoch = len(train_ds) // BATCH_SIZE
epoch_time_est = t_batch * batches_per_epoch
total_time_est_h = epoch_time_est * 200 / 3600

print('=== MEMORY & TIMING SMOKE TEST ===')
print(f'Batch size: {BATCH_SIZE}, accum: {ACCUM_STEPS}, eff BS: {BATCH_SIZE*ACCUM_STEPS}')
print(f'Per-batch time (fwd+bwd): {t_batch*1000:.0f} ms')
print(f'Peak VRAM: {peak_gb:.2f} GB / {total_gb:.1f} GB ({100*peak_gb/total_gb:.0f}% utilization)')
print(f'')
print(f'Projected training budget:')
print(f'  Batches/epoch: {batches_per_epoch}')
print(f'  Est. time/epoch: {epoch_time_est:.0f}s ({epoch_time_est/60:.1f} min)')
print(f'  Est. 200 epochs: {total_time_est_h:.1f} hours')
print()

# Guidance
warnings_issued = False
if peak_gb > total_gb * 0.85:
    print('*** WARNING: Peak VRAM is >85% of GPU capacity. Risk of OOM with larger graphs.')
    print('    Consider: reduce batch_size to 2, or use k=32 truncation.')
    warnings_issued = True
if total_time_est_h > 8:
    print(f'*** WARNING: Estimated {total_time_est_h:.1f}h for 200 epochs exceeds typical Kaggle session (9h).')
    print(f'    Consider: reduce max_epochs to {int(8*3600/epoch_time_est)}, or reduce patience.')
    warnings_issued = True
if not warnings_issued:
    print('All clear — safe to proceed with full training sweep.')

del _model, _optimizer, _batch, _tl

## 10. Training Loop (Kaggle Background Compatible)

**Checkpointing:** Full training state saved every `CKPT_EVERY` epochs to `resume_seed{seed}.pt`:
- Model weights, optimizer state, scheduler state, epoch, best metrics, early-stop counter
- Training auto-resumes from checkpoint if file exists — survives Kaggle session restarts

**Usage:** Set `RUN_SEED` in the next cell. Run one seed per Kaggle session.

In [ ]:
CKPT_EVERY = 5  # save full training state every N epochs

def save_checkpoint(path, model, optimizer, scheduler, epoch, best_val_f1, best_epoch, epochs_no_improve, gate_checked):
    torch.save({
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'epoch': epoch,
        'best_val_f1': best_val_f1,
        'best_epoch': best_epoch,
        'epochs_no_improve': epochs_no_improve,
        'gate_checked': gate_checked,
    }, path)

def load_checkpoint(path, model, optimizer, scheduler):
    ckpt = torch.load(path, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    return (ckpt['epoch'], ckpt['best_val_f1'], ckpt['best_epoch'],
            ckpt['epochs_no_improve'], ckpt['gate_checked'])


def train_single_seed(
    seed, in_dim=14, hidden_dim=128, num_layers=4, num_classes=21,
    num_heads=4, lap_k=8, edge_dim=1,
    max_epochs=200, patience=30, batch_size=4, accum_steps=4, lr=1e-3,
):
    """Train a single seed with full checkpoint/resume support."""
    train_loader, val_loader, test_loader = make_loaders(batch_size)

    # Class weights
    all_labels   = torch.cat([d.y for d in train_ds], dim=0)
    class_counts = torch.bincount(all_labels, minlength=num_classes).float().clamp(min=1)
    cw = 1.0 / class_counts
    cw = (cw / cw.sum() * num_classes).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw, ignore_index=-1)

    best_ckpt_path   = f'best_model_voc_sp_seed{seed}.pt'
    resume_ckpt_path = f'resume_seed{seed}.pt'

    print('=' * 65)
    print(f'PascalVOC-SP | Seed {seed} | BS={batch_size} | Accum={accum_steps} | Eff BS={batch_size*accum_steps}')
    print(f'Checkpointing every {CKPT_EVERY} epochs to {resume_ckpt_path}')
    print('=' * 65)

    set_seed(seed)
    model = HybridGraphFNet_NodeLevel(
        in_dim=in_dim, hidden_dim=hidden_dim, num_layers=num_layers,
        num_classes=num_classes, num_heads=num_heads, lap_k=lap_k,
        edge_dim=edge_dim,
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs, eta_min=1e-5)
    print(f'Parameters: {count_params(model):,}')

    # ---- Resume from checkpoint if it exists ----
    start_epoch = 1
    best_val_f1 = 0.0; best_epoch = 0; epochs_no_improve = 0
    gate_checked = False

    if os.path.exists(resume_ckpt_path):
        print(f'\n>>> RESUMING from {resume_ckpt_path}')
        start_epoch_prev, best_val_f1, best_epoch, epochs_no_improve, gate_checked = \
            load_checkpoint(resume_ckpt_path, model, optimizer, scheduler)
        start_epoch = start_epoch_prev + 1
        print(f'    Resuming at epoch {start_epoch}, best_val_F1={best_val_f1:.4f}@ep{best_epoch}')
        print(f'    epochs_no_improve={epochs_no_improve}, gate_checked={gate_checked}')
    else:
        print('No checkpoint found — starting fresh.')

    if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    for epoch in range(start_epoch, max_epochs + 1):
        # Gate health check at epoch 6
        if epoch == 6 and not gate_checked:
            gc_result = check_gate_health(model, val_loader, device)
            gate_checked = True
            if gc_result:
                print('  Resetting optimizer to lr=5e-4.')
                optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
                scheduler = optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=max_epochs - epoch, eta_min=1e-5)

        # ---- Train epoch ----
        model.train()
        total_loss = 0; optimizer.zero_grad()

        pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                    desc=f'Seed {seed} | Ep {epoch}/{max_epochs}', leave=False)
        for step, batch in pbar:
            batch = batch.to(device)
            logits, mask = model(batch)
            y_dense, _ = to_dense_batch(batch.y, batch.batch, fill_value=-1)

            loss = criterion(logits.reshape(-1, logits.size(-1)),
                             y_dense.reshape(-1).long())
            loss = loss / accum_steps
            loss.backward()
            total_loss += loss.item() * accum_steps

            if (step + 1) % accum_steps == 0 or step == len(train_loader) - 1:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step(); optimizer.zero_grad()
            pbar.set_postfix({'loss': f'{total_loss/(step+1):.4f}'})

        scheduler.step()
        avg_loss = total_loss / len(train_loader)

        # ---- Validate ----
        val_f1 = evaluate_macro_f1(model, val_loader, device)
        improved = ''
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1; best_epoch = epoch; epochs_no_improve = 0
            torch.save(model.state_dict(), best_ckpt_path)
            improved = ' *BEST*'
        else:
            epochs_no_improve += 1

        if epoch % 5 == 0 or improved:
            print(f'  Ep {epoch:>3d} | loss={avg_loss:.4f} | val_F1={val_f1:.4f} | '
                  f'best={best_val_f1:.4f}@ep{best_epoch}{improved}')

        # ---- Periodic checkpoint (full training state) ----
        if epoch % CKPT_EVERY == 0:
            save_checkpoint(resume_ckpt_path, model, optimizer, scheduler,
                           epoch, best_val_f1, best_epoch, epochs_no_improve, gate_checked)
            print(f'  [CKPT] Saved training state at epoch {epoch}')

        # ---- Early stopping ----
        if epochs_no_improve >= patience:
            print(f'  Early stopping at epoch {epoch} (patience={patience})')
            break

    # ---- Final checkpoint ----
    save_checkpoint(resume_ckpt_path, model, optimizer, scheduler,
                   epoch, best_val_f1, best_epoch, epochs_no_improve, gate_checked)

    # ---- Test ----
    elapsed = time.time() - start_time
    model.load_state_dict(torch.load(best_ckpt_path, weights_only=True))
    test_f1 = evaluate_macro_f1(model, test_loader, device)
    peak_mem = torch.cuda.max_memory_allocated()/1e9 if device.type == 'cuda' else 0.0

    print(f'\n  SEED {seed}: test_F1={test_f1:.4f} (val={best_val_f1:.4f}@ep{best_epoch})')
    print(f'  Time this session: {elapsed:.0f}s | Peak VRAM: {peak_mem:.2f} GB')

    result = {'seed': seed, 'test_f1': test_f1, 'best_val_f1': best_val_f1,
              'best_epoch': best_epoch, 'time_s': elapsed, 'peak_gb': peak_mem}

    # ---- Save result to CSV immediately ----
    import csv
    csv_path = 'results_pascalvoc_sp.csv'
    file_exists = os.path.exists(csv_path)
    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['seed','test_f1','best_val_f1','best_epoch','time_s','peak_gb'])
        if not file_exists: writer.writeheader()
        writer.writerow(result)
    print(f'  Result appended to {csv_path}')

    return result

## 11. Run Training (Single Seed)

**Set `RUN_SEED` below.** Run seeds 0, 1, 2 in separate Kaggle sessions.  
If session crashes mid-epoch, just re-run — it resumes from the last checkpoint automatically.

In [ ]:
# ============================
# SET THIS PER KAGGLE SESSION
RUN_SEED = 0  # Change to 1, 2 for subsequent runs
# ============================

result = train_single_seed(
    seed=RUN_SEED,
    in_dim=NODE_FEAT_DIM, hidden_dim=128, num_layers=4,
    num_classes=NUM_CLASSES, num_heads=4, lap_k=8, edge_dim=EDGE_DIM,
    max_epochs=200, patience=30, batch_size=4, accum_steps=4, lr=1e-3,
)

## 12. Baseline Comparison

In [ ]:
import csv

# Load all available results (from this and previous sessions)
csv_path = 'results_pascalvoc_sp.csv'
all_results = []
if os.path.exists(csv_path):
    with open(csv_path, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            all_results.append({k: float(v) if k != 'seed' else int(float(v)) for k, v in row.items()})

f1s = [r['test_f1'] for r in all_results]
n_seeds = len(f1s)

baselines = [
    ('GCN',               '~500k', 0.1268, ''),
    ('GINE',              '~476k', 0.1265, ''),
    ('GatedGCN',          '~509k', 0.2873, '(tuned: 0.3880)'),
    ('Transformer+LapPE', '~488k', 0.2694, ''),
    ('SAN+LapPE',         '~493k', 0.3230, ''),
    ('GPS',               '~500k', 0.3748, '(tuned: 0.4440)'),
]

if f1s:
    mean_f1 = np.mean(f1s)
    std_f1  = np.std(f1s) if n_seeds > 1 else 0.0
    note = f'+/- {std_f1:.4f} ({n_seeds} seed{"s" if n_seeds>1 else ""})'
    baselines.append(('HybridGraphFNet (ours)', f'{count_params(model):,}', mean_f1, note))
else:
    print('No results found in CSV yet.')

print(f'{"Model":<28s} {"Params":<10s} {"Test F1":<10s} Note')
print('-' * 65)
for name, params, f1, note in baselines:
    print(f'{name:<28s} {params:<10s} {f1:<10.4f} {note}')

if n_seeds >= 3:
    print(f'\nFinal result: {mean_f1:.4f} +/- {std_f1:.4f} (3 seeds)')
elif n_seeds > 0:
    print(f'\nPartial result ({n_seeds}/3 seeds): {mean_f1:.4f}')
    print(f'Seeds completed: {[r["seed"] for r in all_results]}')
    print(f'Seeds remaining: {[s for s in [0,1,2] if s not in [r["seed"] for r in all_results]]}')

## 13. Save Results

In [ ]:
# Results are saved incrementally in train_single_seed() — no extra save needed.
# This cell prints the summary for copy-paste into paper_notes.md.

import csv
csv_path = 'results_pascalvoc_sp.csv'
if os.path.exists(csv_path):
    with open(csv_path, 'r') as f:
        reader = csv.DictReader(f)
        all_results = [{k: float(v) if k != 'seed' else int(float(v)) for k, v in row.items()} for row in reader]

    print('--- For paper_notes.md ---')
    print('PascalVOC-SP (node classification, macro F1):')
    f1s = [r['test_f1'] for r in all_results]
    if len(f1s) >= 2:
        print(f'  HybridGraphFNet: {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}')
    for r in all_results:
        print(f'  Seed {r["seed"]:.0f}: F1={r["test_f1"]:.4f} (ep {r["best_epoch"]:.0f}, '
              f'{r["time_s"]:.0f}s, {r["peak_gb"]:.2f} GB)')
else:
    print('No results CSV found. Run training first.')